In [1]:
import pandas as pd
import numpy as np
import datetime
from dateutil.relativedelta import relativedelta
from IPython.display import display

# --- 데이터 로드 ---# HR Core
from services.tables.HR_Core.basic_info_table import emp_df
from services.tables.HR_Core.absence_info_table import absence_info_df
from services.tables.HR_Core.career_info_table import career_info_df
from services.tables.HR_Core.contract_info_table import contract_info_df
from services.tables.HR_Core.department_info_table import department_info_df
from services.tables.HR_Core.job_info_table import job_info_df
from services.tables.HR_Core.position_info_table import position_info_df
from services.tables.HR_Core.pjt_info_table import pjt_info_df
from services.tables.HR_Core.region_info_table import region_info_df
from services.tables.HR_Core.school_info_table import school_info_df
from services.tables.HR_Core.school_table import school_df
from services.tables.HR_Core.job_table import job_df
from services.tables.HR_Core.position_table import position_df
from services.tables.HR_Core.department_table import department_df

# Payroll
from services.tables.Payroll.yearly_payroll_info_table import yearly_payroll_df

# Performance
from services.tables.Performance.evaluation_modified_score_info_table import evaluation_modified_score_df

# Time_Attendance
from services.tables.Time_Attendance.daily_working_info_table import daily_work_info_df
from services.tables.Time_Attendance.detailed_leave_info_table import detailed_leave_info_df
from services.tables.Time_Attendance.leave_type_table import leave_type_df

🔧 [DEV MODE] 개발 모드 활성화:
   - 직원 수: 1000명
   - 날짜 범위: 2020-01-01 ~ 현재
   - 예상 로딩 속도: 프로덕션 수준 (대용량 데이터)


/app/src/services/tables/Time_Attendance/detailed_working_info_table.py:80: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['IS_VACATION'] = df['IS_VACATION'].fillna(False).astype(bool)


In [2]:
today = datetime.datetime.now().date()
today_ts = pd.to_datetime(today)

# --- Feature Engineering ---

# 마스터 테이블의 기반이 될 직원 정보 복사 (+ PERSONAL_ID for age calculation)
master_df = emp_df[['EMP_ID', 'PERSONAL_ID', 'GENDER', 'NATIONALITY', 'IN_DATE', 'OUT_DATE', 'CURRENT_EMP_YN']].copy()

In [3]:
# --- 2.1. Basic Info Features ---
def calculate_age(pid, base_date):
    try:
        birth_year_prefix = {'1': 1900, '2': 1900, '3': 2000, '4': 2000}
        year = birth_year_prefix[pid[7]] + int(pid[:2])
        month = int(pid[2:4])
        day = int(pid[4:6])
        birth_date = datetime.date(year, month, day)
        return base_date.year - birth_date.year - ((base_date.month, base_date.day) < (birth_date.month, birth_date.day))
    except:
        return np.nan

def calculate_age_at_hiring(row):
    try:
        pid = row['PERSONAL_ID']
        in_date = row['IN_DATE']
        birth_year_prefix = {'1': 1900, '2': 1900, '3': 2000, '4': 2000}
        year = birth_year_prefix[pid[7]] + int(pid[:2])
        month = int(pid[2:4])
        day = int(pid[4:6])
        birth_date = pd.to_datetime(datetime.date(year, month, day))
        return (in_date - birth_date).days / 365.25
    except:
        return np.nan

master_df['AGE'] = master_df.apply(lambda row: calculate_age(row['PERSONAL_ID'], today), axis=1)
master_df['TENURE_DAYS'] = (master_df['OUT_DATE'].fillna(today_ts) - master_df['IN_DATE']).dt.days
master_df['IS_LEAVER'] = np.where(master_df['CURRENT_EMP_YN'] == 'N', 1, 0)
master_df['AGE_AT_HIRING'] = master_df.apply(calculate_age_at_hiring, axis=1)
master_df['TENURE_TO_AGE_RATIO'] = (master_df['TENURE_DAYS'] / (master_df['AGE'] * 365.25)).fillna(0)

In [4]:
# --- 2.2. Career Features ---
if not career_info_df.empty:
    career_summary = career_info_df.groupby('EMP_ID').agg(
        PRIOR_CAREER_DAYS=('CAREER_DURATION', 'sum'),
        NUM_PRIOR_COMPANIES=('CAREER_COMPANY_ID', 'nunique'),
        PRIOR_CAREER_RELEVANCE_RATIO=('CAREER_REL_YN', lambda x: (x == 'Y').mean())
    ).reset_index()
    career_summary['AVG_TENURE_PER_COMPANY'] = (career_summary['PRIOR_CAREER_DAYS'] / career_summary['NUM_PRIOR_COMPANIES']).fillna(0)
    master_df = pd.merge(master_df, career_summary, on='EMP_ID', how='left')

In [5]:
# --- 2.3. School Features ---
if not school_info_df.empty:
    school_info_merged = pd.merge(school_info_df, school_df[['SCHOOL_ID', 'SCHOOL_LEVEL']], on='SCHOOL_ID', how='left')
    degree_order = pd.CategoricalDtype(['전문학사', '학사', '석사', '박사'], ordered=True)
    school_info_merged['EDU_DEGREE_CAT'] = school_info_merged['EDU_DEGREE'].astype(degree_order)
    highest_edu = school_info_merged.sort_values('EDU_DEGREE_CAT', ascending=False).groupby('EMP_ID').first().reset_index()
    highest_edu = highest_edu[['EMP_ID', 'EDU_DEGREE', 'SCHOOL_LEVEL', 'MAJOR_CATEGORY']]
    highest_edu.rename(columns={'EDU_DEGREE': 'HIGHEST_DEGREE', 'SCHOOL_LEVEL': 'FINAL_SCHOOL_LEVEL', 'MAJOR_CATEGORY': 'FINAL_MAJOR_CATEGORY'}, inplace=True)
    highest_edu['IS_STEM_MAJOR'] = np.where(highest_edu['FINAL_MAJOR_CATEGORY'] == 'STEM계열', 1, 0)
    master_df = pd.merge(master_df, highest_edu, on='EMP_ID', how='left')

In [6]:
# --- 2.4. Department, Job, Position, Project Features ---
def get_latest_info(df, date_col, info_cols):
    if df.empty or date_col not in df.columns:
        return pd.DataFrame(columns=['EMP_ID'] + info_cols)
    latest = df.sort_values(by=date_col, ascending=False).groupby('EMP_ID').first().reset_index()
    return latest[['EMP_ID'] + info_cols]

latest_dept = get_latest_info(department_info_df, 'DEP_APP_START_DATE', ['DEP_ID', 'TITLE_INFO'])
latest_dept.rename(columns={'DEP_ID': 'LATEST_DEP_ID', 'TITLE_INFO': 'LATEST_TITLE_INFO'}, inplace=True)

latest_job = get_latest_info(job_info_df, 'JOB_APP_START_DATE', ['JOB_ID'])
latest_job.rename(columns={'JOB_ID': 'LATEST_JOB_ID'}, inplace=True)

latest_pos = get_latest_info(position_info_df, 'GRADE_START_DATE', ['POSITION_ID', 'GRADE_ID'])
latest_pos.rename(columns={'POSITION_ID': 'LATEST_POSITION_ID', 'GRADE_ID': 'LATEST_GRADE_ID'}, inplace=True)

master_df = pd.merge(master_df, latest_dept, on='EMP_ID', how='left')
master_df = pd.merge(master_df, latest_job, on='EMP_ID', how='left')
master_df = pd.merge(master_df, latest_pos, on='EMP_ID', how='left')

# [NEW/MODIFIED] Department, Job 계층 구조 처리
# Department: Division/Office 단위로 상위 부서명 사용
dept_map = department_df.set_index('DEP_ID').to_dict('index')
def get_department_hierarchy(dep_id, dept_map):
    hierarchy = {'LATEST_DIVISION_NAME': 'Unknown', 'LATEST_OFFICE_NAME': 'Unknown'}
    if pd.isna(dep_id) or dep_id not in dept_map:
        return pd.Series(hierarchy)

    current_id = dep_id
    for _ in range(10): # 무한 루프 방지를 위한 최대 10단계 탐색
        if pd.isna(current_id) or current_id not in dept_map:
            break
        
        dep_info = dept_map[current_id]
        dep_name = dep_info.get('DEP_NAME', '')

        if 'Division' in dep_name and hierarchy['LATEST_DIVISION_NAME'] == 'Unknown':
            hierarchy['LATEST_DIVISION_NAME'] = dep_name
        if 'Office' in dep_name and hierarchy['LATEST_OFFICE_NAME'] == 'Unknown':
            hierarchy['LATEST_OFFICE_NAME'] = dep_name

        parent_id = dep_info.get('DEP_UPPER_ID')
        if pd.isna(parent_id) or parent_id == current_id:
            break
        current_id = parent_id
    return pd.Series(hierarchy)

dept_hierarchy_cols = master_df['LATEST_DEP_ID'].apply(lambda x: get_department_hierarchy(x, dept_map))
master_df = pd.concat([master_df, dept_hierarchy_cols], axis=1)

# Job: Job Level 1, 2 기준으로 직무명 사용
job_map = job_df.set_index('JOB_ID').to_dict('index')
def get_job_hierarchy(job_id, job_map):
    hierarchy = {'LATEST_JOB_L1_NAME': 'Unknown', 'LATEST_JOB_L2_NAME': 'Unknown'}
    if pd.isna(job_id) or job_id not in job_map:
        return pd.Series(hierarchy)

    # 현재 직무 및 상위 직무 탐색
    current_id = job_id
    for _ in range(10): # 무한 루프 방지
        if pd.isna(current_id) or current_id not in job_map:
            break
        
        job_info = job_map[current_id]
        job_level = job_info.get('JOB_LEVEL')
        job_name = job_info.get('JOB_NAME', 'Unknown')

        if job_level == 1 and hierarchy['LATEST_JOB_L1_NAME'] == 'Unknown':
            hierarchy['LATEST_JOB_L1_NAME'] = job_name
        if job_level == 2 and hierarchy['LATEST_JOB_L2_NAME'] == 'Unknown':
            hierarchy['LATEST_JOB_L2_NAME'] = job_name
        
        parent_id = job_info.get('JOB_UPPER_ID')
        if pd.isna(parent_id) or parent_id == current_id:
            break
        current_id = parent_id
    return pd.Series(hierarchy)

job_hierarchy_cols = master_df['LATEST_JOB_ID'].apply(lambda x: get_job_hierarchy(x, job_map))
master_df = pd.concat([master_df, job_hierarchy_cols], axis=1)

if not department_info_df.empty:
    dept_summary = department_info_df.groupby('EMP_ID').agg(
        NUM_DEP_CHANGES=('DEP_ID', 'nunique'),
        AVG_DEP_TENURE_DAYS=('DEP_DURATION', 'mean')
    ).reset_index()
    last_dept_change_date = department_info_df.groupby('EMP_ID')['DEP_APP_START_DATE'].max().reset_index().rename(columns={'DEP_APP_START_DATE': 'LAST_DEP_CHANGE_DATE'})
    dept_summary = pd.merge(dept_summary, last_dept_change_date, on='EMP_ID', how='left')
    dept_summary['DAYS_SINCE_LAST_DEP_CHANGE'] = (today_ts - dept_summary['LAST_DEP_CHANGE_DATE']).dt.days
    master_df = pd.merge(master_df, dept_summary.drop(columns=['LAST_DEP_CHANGE_DATE']), on='EMP_ID', how='left')

if not position_info_df.empty:
    promotions = position_info_df[position_info_df['CHANGE_REASON'] != 'Initial Assignment']
    promo_summary = promotions.groupby('EMP_ID').agg(
        NUM_PROMOTIONS=('GRADE_ID', 'count'),
        AVG_PROMOTION_SPEED_DAYS=('GRADE_DURATION', 'mean')
    ).reset_index()
    last_promo_date = promotions.groupby('EMP_ID')['GRADE_START_DATE'].max().reset_index().rename(columns={'GRADE_START_DATE': 'LAST_PROMO_DATE'})
    promo_summary = pd.merge(promo_summary, last_promo_date, on='EMP_ID', how='left')
    promo_summary['DAYS_SINCE_LAST_PROMOTION'] = (today_ts - promo_summary['LAST_PROMO_DATE']).dt.days
    promo_summary = pd.merge(promo_summary, master_df[['EMP_ID', 'TENURE_DAYS']], on='EMP_ID', how='left')
    promo_summary['PROMOTION_RATE'] = (promo_summary['NUM_PROMOTIONS'] / (promo_summary['TENURE_DAYS'] / 365.25)).fillna(0)
    master_df = pd.merge(master_df, promo_summary[['EMP_ID', 'NUM_PROMOTIONS', 'AVG_PROMOTION_SPEED_DAYS', 'DAYS_SINCE_LAST_PROMOTION', 'PROMOTION_RATE']], on='EMP_ID', how='left')

if not pjt_info_df.empty:
    pjt_summary = pjt_info_df.groupby('EMP_ID').agg(
        NUM_PROJECTS=('PJT_ID', 'nunique'),
        AVG_PROJECT_DURATION=('PJT_DURATION', 'mean')
    ).reset_index()
    master_df = pd.merge(master_df, pjt_summary, on='EMP_ID', how='left')

In [7]:
# --- 2.5. Payroll Features ---
if not yearly_payroll_df.empty:
    latest_payroll = yearly_payroll_df.sort_values('PAY_YEAR', ascending=False).groupby('EMP_ID').first().reset_index()
    latest_payroll.rename(columns={'TOTAL_PAY': 'LATEST_TOTAL_PAY'}, inplace=True)
    payroll_summary = yearly_payroll_df.groupby('EMP_ID').agg(
        AVG_YOY_GROWTH=('YOY_GROWTH', 'mean'),
        AVG_VARIABLE_PAY_RATIO=('VARIABLE_PAY_RATIO', 'mean')
    ).reset_index()
    master_df = pd.merge(master_df, latest_payroll[['EMP_ID', 'LATEST_TOTAL_PAY']], on='EMP_ID', how='left')
    master_df = pd.merge(master_df, payroll_summary, on='EMP_ID', how='left')

In [8]:
# --- 2.6. Performance Features ---
if not evaluation_modified_score_df.empty:
    latest_eval = evaluation_modified_score_df.sort_values('EVAL_TIME', ascending=False).groupby('EMP_ID').first().reset_index()
    latest_eval.rename(columns={'MODIFIED_SCORE': 'LATEST_EVAL_SCORE'}, inplace=True)
    eval_summary = evaluation_modified_score_df.groupby('EMP_ID').agg(
        AVG_EVAL_SCORE=('MODIFIED_SCORE', 'mean'),
        EVAL_SCORE_STDDEV=('MODIFIED_SCORE', 'std')
    ).reset_index()
    def calculate_trend(group):
        if len(group) < 2:
            return 0
        group['TIME_ORDINAL'] = pd.to_datetime(group['EVAL_TIME'].str.replace('상반기', '-06-01').str.replace('하반기', '-12-01')).map(datetime.datetime.toordinal)
        X = group['TIME_ORDINAL']
        y = group['MODIFIED_SCORE']
        return np.polyfit(X, y, 1)[0] * 365
    eval_trend = evaluation_modified_score_df.groupby('EMP_ID').apply(calculate_trend).reset_index(name='EVAL_SCORE_TREND')
    master_df = pd.merge(master_df, latest_eval[['EMP_ID', 'LATEST_EVAL_SCORE']], on='EMP_ID', how='left')
    master_df = pd.merge(master_df, eval_summary, on='EMP_ID', how='left')
    master_df = pd.merge(master_df, eval_trend, on='EMP_ID', how='left')

/tmp/ipykernel_210/45432619.py:16: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  eval_trend = evaluation_modified_score_df.groupby('EMP_ID').apply(calculate_trend).reset_index(name='EVAL_SCORE_TREND')


In [9]:
# --- 2.7. Time & Attendance Features ---
if not daily_work_info_df.empty:
    ta_summary = daily_work_info_df.groupby('EMP_ID').agg(
        AVG_OVERTIME_MINUTES=('OVERTIME_MINUTES', 'mean'),
        AVG_NIGHT_WORK_MINUTES=('NIGHT_WORK_MINUTES', 'mean')
    ).reset_index()
    master_df = pd.merge(master_df, ta_summary, on='EMP_ID', how='left')

if not detailed_leave_info_df.empty:
    leave_summary = detailed_leave_info_df.groupby('EMP_ID').agg(
        TOTAL_LEAVE_DAYS=('LEAVE_LENGTH', 'sum'),
        AVG_LEAVE_TERM=('LEAVE_LENGTH', 'mean')
    ).reset_index()
    sick_leave_id = leave_type_df[leave_type_df['LEAVE_TYPE_NAME'] == '병휴가']['LEAVE_TYPE_ID'].iloc[0]
    sick_leaves = detailed_leave_info_df[detailed_leave_info_df['LEAVE_TYPE_ID'] == sick_leave_id]
    sick_leave_summary = sick_leaves.groupby('EMP_ID')['LEAVE_LENGTH'].sum().reset_index().rename(columns={'LEAVE_LENGTH': 'SICK_LEAVE_DAYS'})
    leave_summary = pd.merge(leave_summary, sick_leave_summary, on='EMP_ID', how='left')
    leave_summary['SICK_LEAVE_RATIO'] = (leave_summary['SICK_LEAVE_DAYS'] / leave_summary['TOTAL_LEAVE_DAYS']).fillna(0)
    master_df = pd.merge(master_df, leave_summary[['EMP_ID', 'TOTAL_LEAVE_DAYS', 'SICK_LEAVE_RATIO', 'AVG_LEAVE_TERM']], on='EMP_ID', how='left')

In [10]:
# --- 2.8. Absence Features ---
if not absence_info_df.empty:
    absence_summary = absence_info_df.groupby('EMP_ID').agg(
        TOTAL_ABSENCE_DAYS=('ABSENCE_DURATION', 'sum'),
        NUM_ABSENCES=('ABSENCE_ID', 'count')
    ).reset_index()
    master_df = pd.merge(master_df, absence_summary, on='EMP_ID', how='left')

In [11]:
# --- 3. 최종 마스터 테이블 생성 및 저장 ---

# --- 3.1. 데이터 정리 ---
# 날짜 및 임시 ID 데이터 제거
master_df = master_df.drop(columns=['IN_DATE', 'OUT_DATE', 'PERSONAL_ID'])

# Nationality: Korea / Other
master_df['NATIONALITY'] = np.where(master_df['NATIONALITY'] == 'Korea', 'Korea', 'Other')

# 나머지 ID성 컬럼 매핑
pos_name_map = position_df.set_index('POSITION_ID')['POSITION_NAME'].to_dict()
master_df['LATEST_POSITION_NAME'] = master_df['LATEST_POSITION_ID'].map(pos_name_map)

# 결측치 처리
for col in master_df.columns:
    if master_df[col].dtype == 'float64' or master_df[col].dtype == 'float32':
        master_df[col] = master_df[col].fillna(0)
    elif master_df[col].dtype == 'object':
        master_df[col] = master_df[col].fillna('Unknown')

# [MODIFIED] 범주형 변수 인코딩 (One-Hot Encoding)
categorical_cols = [
    'GENDER', 'NATIONALITY', 'HIGHEST_DEGREE', 'FINAL_MAJOR_CATEGORY', 'FINAL_SCHOOL_LEVEL',
    'LATEST_TITLE_INFO', 
    'LATEST_DIVISION_NAME', 'LATEST_OFFICE_NAME', # New
    'LATEST_JOB_L1_NAME', 'LATEST_JOB_L2_NAME', # New
    'LATEST_POSITION_NAME', 
]
categorical_cols_exist = [col for col in categorical_cols if col in master_df.columns]
master_df_encoded = pd.get_dummies(master_df, columns=categorical_cols_exist, drop_first=True)

# 불필요한 ID 컬럼 제거
id_cols_to_drop = ['LATEST_DEP_ID', 'LATEST_JOB_ID', 'LATEST_POSITION_ID', 'LATEST_GRADE_ID']
master_df_encoded = master_df_encoded.drop(columns=[col for col in id_cols_to_drop if col in master_df_encoded.columns])

# --- 3.2. 결과 확인 및 저장 ---
print("마스터 테이블 생성 완료!")
print(f"생성된 테이블의 크기: {master_df_encoded.shape}")
print("--- Columns ---")
print(master_df_encoded.columns.tolist())
print("--- Sample Data ---")
display(master_df_encoded.head())

# 생성된 마스터 테이블을 파일로 저장 (예: CSV)
master_df_encoded.to_csv('ml_master_table.csv', index=False)
print("'ml_master_table.csv' 파일로 저장되었습니다.")

마스터 테이블 생성 완료!
생성된 테이블의 크기: (1000, 67)
--- Columns ---
['EMP_ID', 'CURRENT_EMP_YN', 'AGE', 'TENURE_DAYS', 'IS_LEAVER', 'AGE_AT_HIRING', 'TENURE_TO_AGE_RATIO', 'PRIOR_CAREER_DAYS', 'NUM_PRIOR_COMPANIES', 'PRIOR_CAREER_RELEVANCE_RATIO', 'AVG_TENURE_PER_COMPANY', 'IS_STEM_MAJOR', 'NUM_DEP_CHANGES', 'AVG_DEP_TENURE_DAYS', 'DAYS_SINCE_LAST_DEP_CHANGE', 'NUM_PROMOTIONS', 'AVG_PROMOTION_SPEED_DAYS', 'DAYS_SINCE_LAST_PROMOTION', 'PROMOTION_RATE', 'NUM_PROJECTS', 'AVG_PROJECT_DURATION', 'LATEST_TOTAL_PAY', 'AVG_YOY_GROWTH', 'AVG_VARIABLE_PAY_RATIO', 'LATEST_EVAL_SCORE', 'AVG_EVAL_SCORE', 'EVAL_SCORE_STDDEV', 'EVAL_SCORE_TREND', 'AVG_OVERTIME_MINUTES', 'AVG_NIGHT_WORK_MINUTES', 'TOTAL_LEAVE_DAYS', 'SICK_LEAVE_RATIO', 'AVG_LEAVE_TERM', 'TOTAL_ABSENCE_DAYS', 'NUM_ABSENCES', 'GENDER_M', 'NATIONALITY_Other', 'HIGHEST_DEGREE_박사', 'HIGHEST_DEGREE_석사', 'HIGHEST_DEGREE_전문학사', 'HIGHEST_DEGREE_학사', 'FINAL_MAJOR_CATEGORY_Unknown', 'FINAL_MAJOR_CATEGORY_기타', 'FINAL_MAJOR_CATEGORY_기타공학계열', 'FINAL_MAJOR_CATEG

,EMP_ID,CURRENT_EMP_YN,AGE,TENURE_DAYS,IS_LEAVER,AGE_AT_HIRING,TENURE_TO_AGE_RATIO,PRIOR_CAREER_DAYS,NUM_PRIOR_COMPANIES,PRIOR_CAREER_RELEVANCE_RATIO,...,LATEST_OFFICE_NAME_Global Sales Office,LATEST_OFFICE_NAME_Marketing Office,LATEST_OFFICE_NAME_Production Office,LATEST_OFFICE_NAME_QA Office,LATEST_OFFICE_NAME_R&D Office,LATEST_OFFICE_NAME_Strategy Office,LATEST_OFFICE_NAME_Unknown,LATEST_POSITION_NAME_Director,LATEST_POSITION_NAME_Manager,LATEST_POSITION_NAME_Staff
0,E00001,N,44,2063,1,32.465435,0.128368,817.0,3.0,0.333333,...,False,False,False,False,False,True,False,False,True,False
1,E00002,Y,28,2016,0,23.063655,0.197125,0.0,0.0,0.000000,...,False,False,False,False,False,False,True,False,False,True
2,E00003,Y,44,1409,0,40.755647,0.087673,1870.0,2.0,0.000000,...,False,False,False,False,False,True,False,False,True,False
3,E00004,Y,31,282,0,30.943190,0.024906,409.0,1.0,1.000000,...,False,False,False,False,False,False,False,False,False,True
4,E00005,N,40,1150,1,29.927447,0.078713,0.0,0.0,0.000000,...,False,False,False,False,False,False,True,False,False,True


'ml_master_table.csv' 파일로 저장되었습니다.
